# 第 30 课 · 权重量化：INT8、INT4、缩放因子与误差

**把“4-bit 更省”拆成三个可以验证的问题：数值变成什么？文件怎样存？运行时究竟怎么计算？**

本课从手算量化开始，实现逐张量、逐输出通道、分组量化和真正的 INT4 打包，再比较同一份冻结模型的预测。

| 交付物 | 使用方法 |
|---|---|
| 本 Notebook | 从全新 Python 内核按顺序运行；代码全部内嵌，不导入同名 `.py` |
| `lesson_30_weight_quantization_int8_int4.py` | 同一实现的独立脚本；不需要本 Notebook 或前课源码 |

仅依赖 **PyTorch + Python 标准库**。不下载模型或数据，不要求 GPU，不安装 bitsandbytes。

**关键边界：** 量化文件中的 INT4 确实两值一字节；但本课推理时**先把全部权重反量化为 FP32**。本课验证数值影响和存储格式，不实现低比特矩阵内核，不报告推理加速或 GPU 显存节省。

建议分两次学习：第一次完成第 1–9 节公式、存储与误差；第二次完成模型对照和三个独立练习。

## 0. 从上一课接过来的问题

第 29 课已经区分了“能运行”“格式正确”“任务答对”。量化以后仍然要分别检查这些事情。

本课先用一个**完全离线、可核对**的数值实验建立认识；不会把小模型的结果写成真实 Qwen 的量化成绩。

学习后应当能回答：

1. `weight.to(torch.int8)` 为什么不是完整的量化？
2. `scale`、`zero_point`、范围截断分别做了什么？
3. 逐输出通道和分组到底沿哪条轴？
4. INT4 码保存在 `int8` 中，是否已经只占 4 bit？
5. 权重 MSE、logits 误差、任务正确率为什么不能互相替代？

下面的工具函数只做校验与记录。真正的核心计算会逐步出现。

In [1]:
import argparse
import copy
from dataclasses import dataclass, asdict
import hashlib
import json
import math
from pathlib import Path
import platform
import tempfile

import torch
from torch import nn
from torch.nn import functional as F

SOURCES = {
    "concepts": "https://huggingface.co/docs/transformers/quantization/concept_guide",
    "affine": "https://mlir.llvm.org/docs/Quantization/",
    "round": "https://docs.pytorch.org/docs/2.10/generated/torch.round.html",
    "per_tensor": "https://docs.pytorch.org/docs/2.10/generated/torch.quantize_per_tensor.html",
    "per_channel": "https://docs.pytorch.org/docs/2.10/generated/torch.quantize_per_channel.html",
    "qat": "https://pytorch.org/blog/quantization-aware-training/",
    "bitsandbytes": "https://huggingface.co/docs/transformers/quantization/bitsandbytes",
    "gptq": "https://arxiv.org/abs/2210.17323",
    "awq": "https://arxiv.org/abs/2306.00978",
}
FORMAT = "lesson30-uniform-symmetric-v1"
MODEL_FORMAT = "lesson30-packed-model-v1"


def positive_int(value, name):
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受 bool")


def finite_float(x, name="x"):
    if not isinstance(x, torch.Tensor) or not x.is_floating_point() or x.numel() == 0:
        raise ValueError(f"{name} 必须是非空浮点张量")
    if not bool(torch.isfinite(x).all()):
        raise ValueError(f"{name} 必须全部有限")


def bit_limit(bits):
    if type(bits) is not int or bits not in (4, 8):
        raise ValueError("本课 bits 只能是整数 4 或 8")
    # 为简单的零中心对称范围留出最负值：INT4 [-7,7]，INT8 [-127,127]。
    return 2 ** (bits - 1) - 1


def tensor_bytes(x):
    return x.numel() * x.element_size()


def fingerprint(state):
    h = hashlib.sha256()
    for name, x in sorted(state.items()):
        x = x.detach().cpu().contiguous()
        h.update(f"{name}:{tuple(x.shape)}:{x.dtype}".encode())
        h.update(x.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()


def choose_device(name):
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    if name not in ("cpu", "cuda"):
        raise ValueError("device 只能是 cpu/cuda/auto")
    if name == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用；不会将显式 cuda 请求静默回退到 CPU")
    return torch.device(name)


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

# 本课默认受控 CPU 实验；沿用已有 PyTorch 安装，不需要升级环境。
torch.set_num_threads(1)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
print("本课默认 device: cpu")

Python: 3.13.5
PyTorch: 2.10.0+cpu
CUDA 可用: False
本课默认 device: cpu


## 1. 直接转换类型，会丢掉什么？

先预测 `[-1.0, -0.1, 0.0, 0.1, 1.0]` 转为整数的结果，再运行。

我们的目标不是让所有小数变成零，而是让有限个整数编号代表一组**有刻度的浮点值**。

例如尺度 `s=0.1` 时，整数码 `3` 可以表示实数 `0.3`。尺度就是相邻量化格点之间的距离。

参考：[MLIR 量化定义](https://mlir.llvm.org/docs/Quantization/)。

In [2]:
values = torch.tensor([-1.0, -0.1, 0.0, 0.1, 1.0])
print("原始值:", values.tolist())
print("直接 .to(int8):", values.to(torch.int8).tolist())
print("注意：上面没有保存 scale，也没有约定反量化规则。")

原始值: [-1.0, -0.10000000149011612, 0.0, 0.10000000149011612, 1.0]
直接 .to(int8): [-1, 0, 0, 0, 1]
注意：上面没有保存 scale，也没有约定反量化规则。


## 2. 量化与反量化：先手算，再写函数

本课使用的仿射约定是：

\[
q=\operatorname{clamp}\left(\operatorname{round}(x/s)+z,\ q_{min},q_{max}\right),
\qquad \widehat x=s(q-z)
\]

其中 `s>0` 为尺度，`z` 为表示实数零的整数码。**反量化得到的是近似值，不是保证恢复原始小数。**

对称权重量化取 `z=0`。本课为了保持零中心对称，使用：

- INT8 有效码范围 `[-127,127]`，不用 `-128`。
- INT4 有效码范围 `[-7,7]`，不用 `-8`。

这只是明确的一种约定；有符号 4-bit 的物理范围仍是 `[-8,7]`，不能把两者混为一谈。

最大绝对值量化：令 `a=max(abs(x))`，`Q=2^(bits-1)-1`，则 `s=a/Q`。全零输入选安全尺度 `1`，仍能精确还原零。

参考：[Hugging Face 量化概念](https://huggingface.co/docs/transformers/quantization/concept_guide)、[MLIR 仿射定义](https://mlir.llvm.org/docs/Quantization/)。

In [3]:
def affine_quantize(x, scale, zero_point=0, qmin=-127, qmax=127):
    """已知标量 scale 的教学量化；返回 int32 码，不改变 x。

    本课定义：round(x/s) + z；正好半整数时采用 torch.round 的 ties-to-even。
    对半整数而言，不能未经说明改成 round(x/s + z)（奇数 z 会改变平局）。
    """
    finite_float(x)
    if type(scale) not in (int, float) or not math.isfinite(scale) or scale <= 0:
        raise ValueError("scale 必须为正有限标量")
    if type(qmin) is not int or type(qmax) is not int or not (-32768 <= qmin < qmax <= 32767):
        raise ValueError("整数范围不合法")
    if type(zero_point) is not int or not qmin <= zero_point <= qmax:
        raise ValueError("zero_point 必须是范围内的整数")
    z = torch.round(x / scale) + zero_point
    return z.clamp(qmin, qmax).to(torch.int32)


def affine_dequantize(q, scale, zero_point=0, dtype=torch.float32):
    if not isinstance(q, torch.Tensor) or q.dtype not in (torch.int8, torch.uint8, torch.int16, torch.int32, torch.int64):
        raise ValueError("q 必须是整数张量")
    if type(scale) not in (int, float) or not math.isfinite(scale) or scale <= 0:
        raise ValueError("scale 必须为正有限标量")
    if type(zero_point) is not int or dtype not in (torch.float32, torch.float64):
        raise ValueError("zero_point/dtype 不合法")
    return (q.to(dtype) - zero_point) * scale


def affine_params(x, bits=4):
    """演示无符号仿射量化：范围显式包含实数零，避免全正张量的零点越界。"""
    finite_float(x)
    bit_limit(bits)
    lo = min(float(x.min()), 0.0)
    hi = max(float(x.max()), 0.0)
    if lo == hi:
        return 1.0, 0
    scale = (hi - lo) / (2 ** bits - 1)
    zp = int(torch.round(torch.tensor(-lo / scale, dtype=torch.float64)).item())
    return scale, max(0, min(2 ** bits - 1, zp))


def symmetric_vector(x, bits=4, clip_ratio=1.0):
    finite_float(x)
    limit = bit_limit(bits)
    if type(clip_ratio) not in (int, float) or not 0 < clip_ratio <= 1:
        raise ValueError("clip_ratio 必须在 (0,1] 内")
    bound = float(x.abs().max()) * clip_ratio
    scale = bound / limit if bound else 1.0
    q = affine_quantize(x, scale, 0, -limit, limit)
    y = affine_dequantize(q, scale, 0, torch.float64 if x.dtype == torch.float64 else torch.float32)
    return q, scale, y

In [4]:
for bits in (8, 4):
    q, scale, reconstructed = symmetric_vector(values, bits)
    print(f"{bits}-bit: scale={scale:.9f}")
    print(" 整数码:", q.tolist())
    print(" 重建值:", reconstructed.tolist())
    print(" 最大绝对误差:", float((values - reconstructed).abs().max()))

8-bit: scale=0.007874016
 整数码: [-127, -13, 0, 13, 127]
 重建值: [-1.0, -0.10236220061779022, 0.0, 0.10236220061779022, 1.0]
 最大绝对误差: 0.002362199127674103
4-bit: scale=0.142857143
 整数码: [-7, -1, 0, 1, 7]
 重建值: [-1.0, -0.1428571492433548, 0.0, 0.1428571492433548, 1.0]
 最大绝对误差: 0.04285714775323868


### 2.1 零点不是模型的 bias

固定 `s=0.1, z=5`：实数 `0` 对应整数码 `5`，不是码 `0`。模型原来 Linear 层中的 bias 是另一件事。

`affine_params()` 是一个额外的无符号仿射演示：观察范围显式包含实数零，因此全正、全负或全零输入都能得到合法零点。

本课后续模型仍使用**对称**量化，零点固定为 0，不额外存一张零点表。

In [5]:
x = torch.tensor([-0.2, 0.0, 0.3])
q = affine_quantize(x, scale=0.1, zero_point=5, qmin=0, qmax=15)
print("仿射整数码:", q.tolist())
print("反量化:", affine_dequantize(q, 0.1, 5).tolist())
for sample in (torch.zeros(3), torch.tensor([2.0, 3.0]), torch.tensor([-3.0, -2.0])):
    print("输入:", sample.tolist(), "-> scale, zero_point:", affine_params(sample))

仿射整数码: [3, 5, 8]
反量化: [-0.20000000298023224, 0.0, 0.30000001192092896]
输入: [0.0, 0.0, 0.0] -> scale, zero_point: (1.0, 0)
输入: [2.0, 3.0] -> scale, zero_point: (0.2, 0)
输入: [-3.0, -2.0] -> scale, zero_point: (0.2, 15)


## 3. 舍入误差与截断误差，要分开理解

固定尺度、没有饱和截断时，最近邻舍入的误差不超过 `s/2`（忽略浮点运算误差）。一旦超出表示范围，截断误差可能更大。

PyTorch `round()` 的半整数平局采用 **round-half-to-even**，不是一律向远离零的方向舍入。例如 `2.5→2`，`1.5→2`。

本课固定 `round(x/s)+z` 的运算顺序。极端的半整数平局下，不能未经说明把整数零点挪进 `round()` 内部；两种实现的 tie 规则可能不同。

参考：[torch.round](https://docs.pytorch.org/docs/2.10/generated/torch.round.html)。

In [6]:
ties = torch.tensor([-2.5, -1.5, -0.5, 0.5, 1.5, 2.5])
print("半整数舍入:", torch.round(ties).tolist())
sample = torch.tensor([-2.0, -0.12, 0.0, 0.12, 2.0])
q = affine_quantize(sample, 0.1, 0, -7, 7)
y = affine_dequantize(q, 0.1)
print("固定s=0.1、INT4:", y.tolist())
print("各位置误差:", (sample-y).abs().tolist())
print("不发生截断时的理论界:", 0.1/2)

半整数舍入: [-2.0, -2.0, -0.0, 0.0, 2.0, 2.0]
固定s=0.1、INT4: [-0.699999988079071, -0.10000000149011612, 0.0, 0.10000000149011612, 0.699999988079071]
各位置误差: [1.2999999523162842, 0.019999995827674866, 0.0, 0.019999995827674866, 1.2999999523162842]
不发生截断时的理论界: 0.05


## 4. 一张矩阵用一个尺度，还是多个尺度？

PyTorch Linear 权重形状是 `[out_features,in_features]`，前向是 `X @ W.T + bias`。

| 范围 | 本课的精确定义 | 尺度数量 |
|---|---|---:|
| `tensor` | 整张权重矩阵共享一个尺度 | 1 |
| `channel` | **每个输出行**单独一个尺度 | `out_features` |
| `group` | 每个输出行沿输入轴切成长度为 G 的连续小组 | `out_features × ceil(in_features/G)` |

分组量化沿输入特征轴进行，**不跨到下一行**。末组不足 G 时补零，反量化后裁掉补齐列。

以下封装把码、尺度、形状、分组规则放在同一个包中，避免以后只有整数却不知道怎样恢复。

参考：[PyTorch per-channel 接口](https://docs.pytorch.org/docs/2.10/generated/torch.quantize_per_channel.html)。

In [7]:
def block_layout(shape, scope, group_size=None):
    if not isinstance(shape, (tuple, list)) or len(shape) != 2:
        raise ValueError("weight shape 必须为 [out_features,in_features]")
    out, inc = shape
    positive_int(out, "out_features"); positive_int(inc, "in_features")
    if scope == "tensor":
        if group_size is not None:
            raise ValueError("per-tensor 不接受 group_size")
        return 1, out * inc, inc
    if scope == "channel":
        if group_size is not None:
            raise ValueError("per-channel 不接受 group_size")
        return out, inc, inc
    if scope == "group":
        positive_int(group_size, "group_size")
        padded = math.ceil(inc / group_size) * group_size
        return out * (padded // group_size), group_size, padded
    raise ValueError("scope 只能是 tensor/channel/group")


def quantize_weight(weight, bits=4, scope="group", group_size=32):
    """PTQ：统计仅来自权重，显式 detach；结果在 CPU。

    所有尺度和反量化参考采用 FP32。分组沿每一输出行的输入轴，末组补零。
    此函数还未打包；INT4 的码暂存在 int8 中，仍占每个码 1 字节。
    """
    finite_float(weight, "weight")
    if weight.ndim != 2:
        raise ValueError("weight 必须为二维 Linear 权重")
    limit = bit_limit(bits)
    if scope != "group" and group_size == 32:
        group_size = None  # 允许使用默认调用 quantize_weight(w, 8, 'channel')。
    shape = list(weight.shape)
    count, width, padded = block_layout(shape, scope, group_size)
    w = weight.detach().cpu().float().contiguous()
    finite_float(w, "FP32 weight")
    if scope == "group":
        w = F.pad(w, (0, padded - shape[1]))
    blocks = w.reshape(count, width)
    absmax = blocks.abs().amax(dim=1, keepdim=True)
    scales = torch.where(absmax == 0, torch.ones_like(absmax), absmax / limit)
    if not bool((scales > 0).all()):
        raise ValueError("scale 下溢为零；本课不支持如此小的权重幅值")
    codes = torch.round(blocks / scales).clamp(-limit, limit).to(torch.int8)
    return {"format": FORMAT, "bits": bits, "shape": shape, "scope": scope,
            "group_size": group_size, "packed": False,
            "codes": codes.flatten().clone(), "scales": scales.flatten().clone()}

def packet_codes(packet):
    if not isinstance(packet, dict) or packet.get("format") != FORMAT:
        raise ValueError("未知的量化格式")
    limit = bit_limit(packet["bits"])
    blocks, width, _ = block_layout(packet["shape"], packet["scope"], packet["group_size"])
    n = blocks * width
    if type(packet.get("packed")) is not bool:
        raise ValueError("packed 必须为 bool")
    scale = packet["scales"]
    if not isinstance(scale, torch.Tensor) or scale.dtype != torch.float32 or tuple(scale.shape) != (blocks,):
        raise ValueError("scale 数量或类型错误")
    if not bool(torch.isfinite(scale).all()) or not bool((scale > 0).all()):
        raise ValueError("scale 必须正且有限")
    if packet["packed"]:
        if packet["bits"] != 4:
            raise ValueError("本课只为4-bit执行 nibble 打包")
        codes = unpack_int4(packet["codes"], n)
    else:
        codes = packet["codes"]
        if not isinstance(codes, torch.Tensor) or codes.dtype != torch.int8 or tuple(codes.shape) != (n,):
            raise ValueError("未打包 codes 数量或类型错误")
    if codes.device != scale.device:
        raise ValueError("codes 与 scales 的 device 不同")
    if bool(((codes < -limit) | (codes > limit)).any()):
        raise ValueError("码超出本课对称量化范围（最负值被保留）")
    if packet["scope"] == "group":
        out, inc = packet["shape"]
        pad_cols = (n // out) - inc
        if pad_cols and bool((codes.reshape(out, -1)[:, inc:] != 0).any()):
            raise ValueError("分组补齐部分必须为零码")
    return codes


def dequantize_weight(packet):
    codes = packet_codes(packet)
    blocks, width, padded = block_layout(packet["shape"], packet["scope"], packet["group_size"])
    w = codes.float().reshape(blocks, width) * packet["scales"].reshape(blocks, 1)
    out, inc = packet["shape"]
    return w.reshape(out, padded)[:, :inc].contiguous()

### 4.1 离群值：一行大数，会不会影响另一行小数？

下面第一行在 `0.1–0.4`，第二行在 `10–40`。先预测：共享最大值 `40` 的 INT4 尺度，会怎样处理第一行？

这个例子不是说逐通道一定在所有任务中更好，而是展示**尺度共享范围的实际影响**。

In [8]:
w = torch.tensor([[0.1, 0.2, 0.3, 0.4], [10.0, 20.0, 30.0, 40.0]])
for scope, group in (("tensor", None), ("channel", None), ("group", 2)):
    packet = quantize_weight(w, bits=4, scope=scope, group_size=group)
    print(scope, "scales:", packet["scales"].tolist())
    print(dequantize_weight(packet))
    print("MSE:", float((dequantize_weight(packet)-w).square().mean()))

tensor scales: [5.714285850524902]
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000],
        [11.4286, 22.8571, 28.5714, 40.0000]])
MSE: 1.5681121349334717
channel scales: [0.05714285746216774, 5.714285850524902]
tensor([[ 0.1143,  0.2286,  0.2857,  0.4000],
        [11.4286, 22.8571, 28.5714, 40.0000]])
MSE: 1.530765175819397
group scales: [0.02857142873108387, 0.05714285746216774, 2.857142925262451, 5.714285850524902]
tensor([[ 0.1143,  0.2000,  0.2857,  0.4000],
        [11.4286, 20.0000, 28.5714, 40.0000]])
MSE: 0.5102546215057373


### 4.2 非整除形状：必须记录补齐规则

`[2,5]` 的权重按每组 4 个数切分时，每行需要 2 组，总共保存 16 个码，而不是原来的 10 个码。填入的零不改变对称 absmax 尺度，但它们在本课格式中仍占空间。

不要把 `original_numel × bits/8` 当作所有格式的精确字节数。

In [9]:
ragged_weight = torch.arange(10, dtype=torch.float32).reshape(2, 5) / 10
packet = quantize_weight(ragged_weight, 4, "group", 4)
print("原始形状:", packet["shape"])
print("码数量:", packet["codes"].numel())
print("尺度数量:", packet["scales"].numel())
print("恢复形状:", list(dequantize_weight(packet).shape))
print("末组补齐后整数码:"); print(packet["codes"].reshape(2, 8))

原始形状: [2, 5]
码数量: 16
尺度数量: 4
恢复形状: [2, 5]
末组补齐后整数码:
tensor([[0, 2, 5, 7, 7, 0, 0, 0],
        [4, 5, 6, 7, 7, 0, 0, 0]], dtype=torch.int8)


## 5. 真正的 4-bit 存储：把两个码装进一个字节

目前的 INT4 码虽然数值范围很小，却放在 `torch.int8` 中：**每个码仍占 1 字节**。

本课定义一种清楚、可往返的打包格式：

1. 将有符号码 `q∈[-8,7]` 映射为 `u=q+8∈[0,15]`。
2. 第一个值放低 4 位，第二个值放高 4 位。
3. `packed = low | (high << 4)`。
4. 奇数个码的末尾高 4 位写入 `8`，代表补齐的零；解包时根据原数量裁掉。

这是**偏移二进制教学格式，不是补码编码**。不能把产物直接交给任意外部 INT4 内核。

`pack_int4()` 支持物理范围中的全部16个码；对称权重量化自身只使用其中15个。

In [10]:
def pack_int4(codes):
    """有符号 [-8,7] → 偏移二进制 [0,15]；先出现的码放低4位。

    这是教学文件约定，不是补码 INT4，也不保证兼容外部内核。
    奇数个码的空高4位写入 8，即解码后的零。
    """
    if not isinstance(codes, torch.Tensor) or codes.dtype not in (torch.int8, torch.int16, torch.int32, torch.int64):
        raise ValueError("codes 必须是有符号整数张量")
    flat = codes.reshape(-1)
    if flat.numel() and bool(((flat < -8) | (flat > 7)).any()):
        raise ValueError("INT4 码必须在 [-8,7]")
    u = (flat.to(torch.int16) + 8).to(torch.uint8)
    if u.numel() % 2:
        u = torch.cat((u, torch.tensor([8], dtype=torch.uint8, device=u.device)))
    return u[0::2] | (u[1::2] << 4)


def unpack_int4(packed, count):
    if not isinstance(packed, torch.Tensor) or packed.dtype != torch.uint8 or packed.ndim != 1:
        raise ValueError("packed 必须是一维 uint8")
    if type(count) is not int or count < 0 or packed.numel() != (count + 1) // 2:
        raise ValueError("打包长度与码数量不符")
    if count % 2 and int(packed[-1] >> 4) != 8:
        raise ValueError("奇数末尾的填充 nibble 不合法")
    u = torch.empty(packed.numel() * 2, dtype=torch.int16, device=packed.device)
    u[0::2] = (packed & 15).to(torch.int16)
    u[1::2] = (packed >> 4).to(torch.int16)
    return (u[:count] - 8).to(torch.int8)


def pack_packet(packet):
    codes = packet_codes(packet)
    result = copy.deepcopy(packet)
    if packet["bits"] == 4:
        result["codes"] = pack_int4(codes)
        result["packed"] = True
    return result


def packet_storage(packet):
    packet_codes(packet)
    code_bytes = tensor_bytes(packet["codes"])
    scale_bytes = tensor_bytes(packet["scales"])
    n = math.prod(packet["shape"])
    return {"original_values": n, "code_bytes": code_bytes, "scale_bytes": scale_bytes,
            "tensor_payload_bytes": code_bytes + scale_bytes,
            "effective_bits_per_original_weight": 8 * (code_bytes + scale_bytes) / n,
            "note": "含分组补齐与量化尺度；不含 Python/文件头元数据，不是峰值显存"}

In [11]:
codes = torch.tensor([-7, -1, 0, 7, 3], dtype=torch.int8)
packed = pack_int4(codes)
print("原始码:", codes.tolist())
print("打包字节:", packed.tolist())
print("二进制:", [format(int(x), "08b") for x in packed])
print("解包:", unpack_int4(packed, codes.numel()).tolist())
print("实际字节:", tensor_bytes(codes), "→", tensor_bytes(packed))
assert torch.equal(codes, unpack_int4(packed, codes.numel()))

原始码: [-7, -1, 0, 7, 3]
打包字节: [113, 248, 139]
二进制: ['01110001', '11111000', '10001011']
解包: [-7, -1, 0, 7, 3]
实际字节: 5 → 3


## 6. 4-bit 不一定刚好是每个权重 4 bit

无补齐、每组 G 个权重、每个尺度用 FP32（32 bit）、零点固定不存时：

\[
\text{有效位数/权重}=4+\frac{32}{G}
\]

例如 `G=32`，实际张量载荷是 **5 bit/权重**。`G=16` 则是 **6 bit/权重**。

下面统一使用 `[32,128]`，没有分组补齐，分别统计码与尺度。Python 对象、配置字符串和文件归档头还没有算进去。

**逻辑载荷、文件大小、模型运行权重占用、GPU峰值显存，是四个不同指标。**

In [12]:
def error_metrics(original, reconstructed):
    if original.shape != reconstructed.shape:
        raise ValueError("误差比较的形状不同")
    a = original.detach().double()
    delta = reconstructed.detach().double() - a
    mse = float(delta.square().mean())
    denom = float(a.square().sum())
    return {"mse": mse, "mae": float(delta.abs().mean()), "max_abs": float(delta.abs().max()),
            "relative_l2": math.sqrt(float(delta.square().sum()) / denom) if denom else None}


def numeric_experiments():
    values = torch.tensor([-1.0, -0.1, 0.0, 0.1, 1.0])
    basic = {}
    for bits in (8, 4):
        q, scale, restored = symmetric_vector(values, bits)
        basic[str(bits)] = {"scale": scale, "codes": q.tolist(), "restored": restored.tolist(),
                            "error": error_metrics(values, restored)}
    # 通道之间量级不同：全张量尺度会让小幅值行失去分辨率。
    w = torch.tensor([[0.1, 0.2, 0.3, 0.4], [10.0, 20.0, 30.0, 40.0]])
    scopes = {}
    for scope, group in (("tensor", None), ("channel", None), ("group", 2)):
        p = quantize_weight(w, 4, scope, group)
        r = dequantize_weight(p)
        scopes[scope] = {"scales": p["scales"].tolist(), "restored": r.tolist(), "error": error_metrics(w, r)}
    signed = torch.tensor([-7, -1, 0, 7, 3], dtype=torch.int8)
    packed = pack_int4(signed)
    storage_w = torch.linspace(-1, 1, 4096).reshape(32, 128)
    storage = {}
    for scope, group in (("channel", None), ("group", 32), ("group", 16)):
        p = quantize_weight(storage_w, 4, scope, group)
        storage[f"{scope}_{group}"] = {"unpacked": packet_storage(p), "packed": packet_storage(pack_packet(p))}
    pos = torch.tensor([0.0, 0.5, 1.0])
    s, z = affine_params(pos, 4)
    aq = affine_quantize(pos, s, z, 0, 15)
    # 固定 scale=0.1, z=5 展示整数5对应实数零（不是 q=0）。
    shifted = affine_quantize(torch.tensor([-0.2, 0.0, 0.3]), 0.1, 5, 0, 15)
    return {"basic": basic, "row_scale_demo": scopes,
            "packing": {"codes": signed.tolist(), "bytes": packed.tolist(),
                        "restored": unpack_int4(packed, signed.numel()).tolist()},
            "storage": storage,
            "affine": {"scale": s, "zero_point": z, "codes": aq.tolist(),
                       "restored": affine_dequantize(aq, s, z).tolist(), "shifted_codes": shifted.tolist()},
            "round_even": torch.round(torch.tensor([-2.5, -1.5, -0.5, 0.5, 1.5, 2.5])).tolist(),
            "naive_int8_cast": values.to(torch.int8).tolist()}

numeric = numeric_experiments()
for name, row in numeric["storage"].items():
    print(name, "未打包:", row["unpacked"]["tensor_payload_bytes"],
          "打包:", row["packed"]["tensor_payload_bytes"],
          "bits/weight:", row["packed"]["effective_bits_per_original_weight"])
print("同一张矩阵FP32载荷:", 32*128*4, "字节")

channel_None 未打包: 4224 打包: 2176 bits/weight: 4.25
group_32 未打包: 4608 打包: 2560 bits/weight: 5.0
group_16 未打包: 5120 打包: 3072 bits/weight: 6.0
同一张矩阵FP32载荷: 16384 字节


## 7. 权重误差小，不代表真实输出误差一定更小

对线性层：

\[
Y=XW^T+b,\quad \widehat Y=X\widehat W^T+b,\quad
\widehat Y-Y=X(\widehat W-W)^T
\]

输出误差与输入 X 有关，不能只看 `mean((W-W_hat)^2)`。

下面用一个构造性例子：权重最后一维是大值 `8`，但校准输入的最后一维总为 0。比较预先规定的两种截断阈值，只用校准输入选择。随后**单独查看分布改变的输入**，不根据它倒过来改选择。

这只是校准思想实验，**不是 AWQ 或 GPTQ 实现**。这些方法会使用更复杂的激活或二阶信息。

参考：[AWQ 原论文](https://arxiv.org/abs/2306.00978)、[GPTQ 原论文](https://arxiv.org/abs/2210.17323)。

In [13]:
def calibration_demo():
    """两个预先固定候选；只用 calibration 选，不看 shift_test 选择。

    它不是 AWQ/GPTQ 的实现，仅说明输入分布会影响量化输出误差。
    """
    w = torch.tensor([[0.09, 0.12, 0.18, 8.0]])
    calibration = torch.tensor([[1.0, 2.0, 1.0, 0.0]])
    shift_test = torch.tensor([[0.0, 0.0, 0.0, 1.0]])
    rows = []
    for ratio in (1.0, 0.1):
        q, scale, restored = symmetric_vector(w, 4, ratio)
        rows.append({"clip_ratio": ratio, "scale": scale, "restored": restored.tolist(),
                     "weight_mse": error_metrics(w, restored)["mse"],
                     "calibration_mse": error_metrics(calibration @ w.T, calibration @ restored.T)["mse"],
                     "shift_test_mse": error_metrics(shift_test @ w.T, shift_test @ restored.T)["mse"],
                     "clipped_count": int((w.abs() > float(w.abs().max()) * ratio).sum())})
    selected = min(range(len(rows)), key=lambda i: rows[i]["calibration_mse"])
    return {"candidates": rows, "selected_by_calibration_only": selected,
            "scope": "单个构造性输入，展示分布偏移风险；不作泛化保证"}

calibration = calibration_demo()
for row in calibration["candidates"]:
    print(json.dumps(row, ensure_ascii=False, indent=2))
print("只按校准误差选中的候选:", calibration["selected_by_calibration_only"])

{
  "clip_ratio": 1.0,
  "scale": 1.1428571428571428,
  "restored": [
    [
      0.0,
      0.0,
      0.0,
      8.0
    ]
  ],
  "weight_mse": 0.013725000643730181,
  "calibration_mse": 0.26009999027252206,
  "shift_test_mse": 0.0,
  "clipped_count": 0
}
{
  "clip_ratio": 0.1,
  "scale": 0.1142857142857143,
  "restored": [
    [
      0.11428571492433548,
      0.11428571492433548,
      0.22857142984867096,
      0.800000011920929
    ]
  ],
  "weight_mse": 12.960745365060074,
  "calibration_mse": 0.0037734736977803607,
  "shift_test_mse": 51.83999982833862,
  "clipped_count": 1
}
只按校准误差选中的候选: 1


### 从这个反例应当得出什么？

在当前构造里，保留全范围会把几个小权重舍入成零；截断大权重能改善校准输入的输出。但当新输入恰好用到被截断的那一维，误差会很大。

因此，校准数据应该与目标使用分布相关；不能用测试题答案或测试效果反复挑阈值，再把相同题目称作独立测试。

本课完整模型实验**不做激活校准搜索**：scale 只由权重 absmax 决定，六种量化方案事先固定。

## 8. logits 误差很小，也可能改掉下一 token

设两候选原始分数相差很小。量化后只要两者顺序反转，贪心选出的 token 就会变化。后续生成前缀一旦不同，生成路径也可能不同。

这不要求出现很大的权重 MSE。先看两个分数的最小反例。

In [14]:
before = torch.tensor([0.501, 0.499])
after = torch.tensor([0.499, 0.501])
print("最大分数误差:", float((after-before).abs().max()))
print("贪心编号:", int(before.argmax()), "→", int(after.argmax()))
print("原始第一/二名间隔:", float(before[0]-before[1]))
print("这只是构造的logits扰动例子，不是某个真实模型的量化结果。")

最大分数误差: 0.001999974250793457
贪心编号: 0 → 1
原始第一/二名间隔: 0.001999974250793457
这只是构造的logits扰动例子，不是某个真实模型的量化结果。


## 9. PTQ、QAT、混合精度与 NF4，不要合并成一个词

| 名称 | 本课怎样区分 |
|---|---|
| 本课 PTQ | 模型先训练，再冻结；根据已有权重确定 scale 和整数码 |
| QAT | 训练过程中模拟量化误差，并采用合适的梯度近似；本课没有实现 |
| 第24课 AMP | 让部分算子使用低精度浮点计算；不是把全部权重压成整数码 |
| NF4 / FP4 | 其他4-bit数值格式；不等于本课的均匀 INT4 格点 |
| 低比特推理内核 | 能按指定存储布局加载、解码或计算的实现；本课不提供 |

普通 `round()` 的自动梯度不能直接代替 QAT 中的直通估计器。下面亲自观察。完整模型量化函数显式 `detach()`，不假装量化本身可以照常训练。

参考：[PyTorch QAT 说明](https://pytorch.org/blog/quantization-aware-training/)、[bitsandbytes 格式与计算精度](https://huggingface.co/docs/transformers/quantization/bitsandbytes)。

In [15]:
v = torch.tensor([0.2, 0.7], requires_grad=True)
torch.round(v).sum().backward()
print("普通round的梯度:", v.grad)
print("这不是STE；本课不使用它来训练整数权重。")

普通round的梯度: tensor([0., 0.])
这不是STE；本课不使用它来训练整数权重。


## 10. 接回完整模型：只训练一次，然后冻结

为了隔离量化影响，使用一个两层小型解码器：RoPE、GQA、RMSNorm、SwiGLU，隐藏维度48。

数据是固定的符号序列：`[BOS,a,b,a,b,a,b,a,b,EOS]`。12个符号组成132种不同有序对，固定切分 **106条训练、26条测试**。本课不是文本聊天，也不执行代码。

- 固定100次参数更新，只训练一个 FP32 源模型；不依测试集选择最佳轮次。
- 然后冻结它，复制出预先规定的6种量化变体，不再训练。
- 量化所有 `nn.Linear.weight`，包括词表输出头；Embedding、归一化保持 FP32。
- 激活、注意力和反量化后的矩阵计算都使用 FP32；不是 W4A16 内核。
- 本课小模型不提供 KV cache，不把权重量化与缓存量化混为一谈。

这部分沿用前几课的模块；可折叠代码仅为让讲义独立运行，不需要先重新抄写整套模型。

In [16]:
@dataclass
class ModelConfig:
    vocab_size: int = 15
    dim: int = 48
    layers: int = 2
    heads: int = 4
    kv_heads: int = 2
    intermediate: int = 128
    max_len: int = 16


class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.square().mean(-1, keepdim=True) + 1e-6) * self.weight


def rope(x):
    d = x.shape[-1]
    freq = torch.arange(0, d, 2, device=x.device, dtype=torch.float32)
    freq = 10000.0 ** (-freq / d)
    angle = torch.arange(x.shape[-2], device=x.device).float()[:, None] * freq[None, :]
    even, odd = x[..., 0::2], x[..., 1::2]
    c, s = angle.cos().to(x.dtype), angle.sin().to(x.dtype)
    return torch.stack((even * c - odd * s, even * s + odd * c), dim=-1).flatten(-2)


class Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        hd = c.dim // c.heads
        self.q_proj = nn.Linear(c.dim, c.dim, bias=False)
        self.k_proj = nn.Linear(c.dim, c.kv_heads * hd, bias=False)
        self.v_proj = nn.Linear(c.dim, c.kv_heads * hd, bias=False)
        self.o_proj = nn.Linear(c.dim, c.dim, bias=False)

    def forward(self, x):
        b, t, d = x.shape
        c = self.c; hd = d // c.heads
        q = rope(self.q_proj(x).reshape(b, t, c.heads, hd).transpose(1, 2))
        k = rope(self.k_proj(x).reshape(b, t, c.kv_heads, hd).transpose(1, 2))
        v = self.v_proj(x).reshape(b, t, c.kv_heads, hd).transpose(1, 2)
        k = k.repeat_interleave(c.heads // c.kv_heads, 1)
        v = v.repeat_interleave(c.heads // c.kv_heads, 1)
        scores = q @ k.transpose(-1, -2) / math.sqrt(hd)
        allowed = torch.ones(t, t, device=x.device, dtype=torch.bool).tril()
        weights = scores.masked_fill(~allowed, float("-inf")).softmax(-1)
        h = (weights @ v).transpose(1, 2).contiguous().reshape(b, t, d)
        return self.o_proj(h)


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.norm1 = RMSNorm(c.dim); self.norm2 = RMSNorm(c.dim)
        self.attn = Attention(c)
        self.gate = nn.Linear(c.dim, c.intermediate, bias=False)
        self.up = nn.Linear(c.dim, c.intermediate, bias=False)
        self.down = nn.Linear(c.intermediate, c.dim, bias=False)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        h = self.norm2(x)
        return x + self.down(F.silu(self.gate(h)) * self.up(h))


class TinyDecoder(nn.Module):
    def __init__(self, c=ModelConfig()):
        super().__init__()
        for key, value in asdict(c).items(): positive_int(value, key)
        if c.dim % c.heads or c.heads % c.kv_heads or (c.dim // c.heads) % 2:
            raise ValueError("模型头数、维度不兼容")
        self.config = c
        self.embedding = nn.Embedding(c.vocab_size, c.dim)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.layers)])
        self.norm = RMSNorm(c.dim)
        self.head = nn.Linear(c.dim, c.vocab_size, bias=False)

    def forward(self, ids):
        if ids.ndim != 2 or ids.dtype != torch.long or not 1 <= ids.shape[1] <= self.config.max_len:
            raise ValueError("ids 应是合法长度的 [B,T] LongTensor")
        h = self.embedding(ids)
        for block in self.blocks: h = block(h)
        return self.head(self.norm(h))


def new_model(seed=3001):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        return TinyDecoder()


def make_data():
    # 固定一套数据；ID0为保留PAD，1=BOS，2=EOS，3..14为12个符号。
    rows = [[1, a, b, a, b, a, b, a, b, 2]
            for a in range(3, 15) for b in range(3, 15) if a != b]
    all_ids = torch.tensor(rows, dtype=torch.long)
    g = torch.Generator().manual_seed(3002)
    order = torch.randperm(len(rows), generator=g)
    train, test = all_ids[order[:106]], all_ids[order[106:]]
    return {"train": train, "test": test, "order": order}


def evaluate(model, rows):
    old = model.training
    model.eval()
    device = next(model.parameters()).device
    try:
        with torch.no_grad():
            x, y = rows[:, :-1].to(device), rows[:, 1:].to(device)
            logits = model(x)
            token_nll = F.cross_entropy(logits.flatten(0, 1), y.flatten(), reduction="none").reshape_as(y)
            loss = token_nll.mean()
            pred = logits.argmax(-1)
            return {"loss": float(loss), "accuracy": float((pred == y).float().mean()),
                    "logits": logits.cpu(), "predictions": pred.cpu(), "targets": y.cpu(),
                    "token_nll": token_nll.cpu()}
    finally:
        model.train(old)


def train_source(steps=100, device=torch.device("cpu")):
    positive_int(steps, "steps")
    model = new_model().to(device)
    data = make_data()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
    g = torch.Generator().manual_seed(3003)
    history = []
    model.train()
    for step in range(steps):
        indices = torch.randperm(len(data["train"]), generator=g)[:16]
        rows = data["train"][indices].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(rows[:, :-1])
        loss = F.cross_entropy(logits.flatten(0, 1), rows[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
        optimizer.step()
        history.append({"step": step + 1, "train_batch_loss": float(loss.detach())})
    # 固定最后一轮；test 不参与步数、量化参数或阈值选择。
    model.eval().requires_grad_(False)
    return model, data, history

In [17]:
shape_probe = new_model()
print("模型参数量:", sum(p.numel() for p in shape_probe.parameters()))
print("固定数据规模:", {k: len(v) for k,v in make_data().items() if k != "order"})
print("输入/标签形状:", make_data()["test"][:, :-1].shape, make_data()["test"][:, 1:].shape)
del shape_probe

模型参数量: 52368
固定数据规模: {'train': 106, 'test': 26}
输入/标签形状: torch.Size([26, 9]) torch.Size([26, 9])


## 11. 保存紧凑权重，再严格重建 FP32 参考模型

量化包保存 `bits、scope、group_size、shape、codes、scales`。模型包还保存配置、未量化状态和源模型指纹。

恢复时先核对字段、码范围、尺度数量和正有限性，再严格加载。缺失一层或改错分组规格，都应报错，而不是只因张量数量大致相同就继续运行。

**重要：** `restore_model()` 会完整反量化并创建 FP32 参数。文件确实变小，但推理模型的权重载荷回到 FP32 大小；它只是数值参考执行器。

保存格式是本课自定义格式，不兼容 bitsandbytes/GGUF/GPTQ/AWQ。只加载可信来源的文件。

In [18]:
def pack_model(model, bits=4, scope="group", group_size=32):
    if scope != "group" and group_size == 32:
        group_size = None
    before = fingerprint(model.state_dict())
    linear_weights = {f"{name}.weight": mod.weight for name, mod in model.named_modules() if isinstance(mod, nn.Linear)}
    packets = {}
    for name, w in linear_weights.items():
        packets[name] = pack_packet(quantize_weight(w, bits, scope, group_size))
    float_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items() if name not in packets}
    if fingerprint(model.state_dict()) != before:
        raise RuntimeError("量化不应修改源模型")
    return {"format": MODEL_FORMAT, "config": asdict(model.config), "source_sha256": before,
            "quantized_weights": packets, "float_state": float_state,
            "quantization": {"bits": bits, "scope": scope, "group_size": group_size,
                             "scheme": "symmetric-narrow-range", "compute": "dequantize-all-to-fp32"}}


def restore_model(payload, device=torch.device("cpu")):
    """从紧凑文件恢复FP32数值参考，NOT 低比特执行器。只加载可信来源。"""
    if not isinstance(payload, dict) or payload.get("format") != MODEL_FORMAT:
        raise ValueError("不兼容的教学模型格式")
    with torch.random.fork_rng(devices=[]):
        model = TinyDecoder(ModelConfig(**payload["config"]))
    expected = model.state_dict()
    linear_names = {f"{name}.weight" for name, mod in model.named_modules() if isinstance(mod, nn.Linear)}
    qw, fs = payload["quantized_weights"], payload["float_state"]
    if set(qw) != linear_names or set(fs) != set(expected) - linear_names:
        raise ValueError("模型参数缺失、重复或存在未知参数")
    state = {}
    meta = payload["quantization"]
    for name, spec in expected.items():
        if name in qw:
            packet = qw[name]
            if packet["bits"] != meta["bits"] or packet["scope"] != meta["scope"] or packet["group_size"] != meta["group_size"]:
                raise ValueError("模型与权重包的量化元数据不一致")
            value = dequantize_weight(packet)
        else:
            value = fs[name]
        if not isinstance(value, torch.Tensor) or value.dtype != torch.float32 or value.shape != spec.shape or not bool(torch.isfinite(value).all()):
            raise ValueError(f"{name} 的形状、类型或有限性不匹配")
        state[name] = value
    model.load_state_dict(state, strict=True)
    return model.to(device).eval().requires_grad_(False)


def model_payload_storage(payload):
    details = [packet_storage(p) for p in payload["quantized_weights"].values()]
    codes = sum(p["code_bytes"] for p in details)
    scales = sum(p["scale_bytes"] for p in details)
    unquantized = sum(tensor_bytes(t) for t in payload["float_state"].values())
    n = sum(p["original_values"] for p in details)
    return {"quantized_weight_values": n, "code_bytes": codes, "scale_bytes": scales,
            "unquantized_tensor_bytes": unquantized, "total_tensor_payload_bytes": codes + scales + unquantized,
            "note": "仅存储张量载荷；不含文件头/对象开销；恢复推理会把全部权重转回FP32"}


def compare_predictions(base, other):
    bp, qp, target = base["predictions"], other["predictions"], base["targets"]
    if not torch.equal(target, other["targets"]):
        raise ValueError("配对评测必须使用相同 targets")
    b_ok, q_ok = bp == target, qp == target
    delta = other["logits"] - base["logits"]
    return {"tokens": target.numel(), "changed_top1": int((bp != qp).sum()),
            "top1_agreement": float((bp == qp).float().mean()),
            "fixed_tokens": int((~b_ok & q_ok).sum()), "regressed_tokens": int((b_ok & ~q_ok).sum()),
            "logits_max_abs": float(delta.abs().max()), "logits_rmse": float(delta.square().mean().sqrt())}

## 12. 运行六种预先固定的量化方案

下面是完整流程。输出目录是当前工作目录下的 `lesson_30_outputs`，同名生成文件会覆盖。它不修改前几课的文件。

`predictions.json` 保留全部测试目标、各方案的逐 token 预测和逐 token 负对数似然。`metrics.json` 汇总 loss、准确率、改对/改错数量、logits 误差与载荷。`weight_errors.json` 保存每个矩阵的误差。

不能把本课26条合成测试序列的结果，移植为第29课24道真实题目的分数。

In [19]:
VARIANTS = (
    ("int8_tensor", 8, "tensor", None),
    ("int8_channel", 8, "channel", None),
    ("int4_tensor", 4, "tensor", None),
    ("int4_channel", 4, "channel", None),
    ("int4_group32", 4, "group", 32),
    ("int4_group16", 4, "group", 16),
)


def run_experiment(output_dir="lesson_30_outputs", steps=100, device=torch.device("cpu")):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    math_results = {"numeric": numeric_experiments(), "calibration": calibration_demo()}
    write_json(out / "numerics.json", math_results)
    model, data, history = train_source(steps, device)
    source_hash = fingerprint(model.state_dict())
    base = evaluate(model, data["test"])
    base_bytes = sum(tensor_bytes(t) for t in model.state_dict().values())
    src = {"config": asdict(model.config), "state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
           "source_sha256": source_hash, "note": "推理权重；不是断点续训检查点"}
    torch.save(src, out / "source_fp32.pt")
    results = [{"variant": "fp32", "loss": base["loss"], "accuracy": base["accuracy"],
                "fp32_runtime_parameter_bytes": base_bytes, "total_tensor_payload_bytes": base_bytes,
                "file_bytes": (out / "source_fp32.pt").stat().st_size, "changed_top1": 0,
                "fixed_tokens": 0, "regressed_tokens": 0, "top1_agreement": 1.0,
                "logits_max_abs": 0.0, "logits_rmse": 0.0}]
    raw = {"targets": base["targets"].tolist(), "predictions": {"fp32": base["predictions"].tolist()},
           "token_nll": {"fp32": base["token_nll"].tolist()}}
    weight_errors = {}
    for name, bits, scope, group in VARIANTS:
        payload = pack_model(model, bits, scope, group)
        path = out / f"{name}.pt"
        torch.save(payload, path)
        loaded = torch.load(path, map_location="cpu", weights_only=True)
        reconstructed = restore_model(loaded, device)
        other = evaluate(reconstructed, data["test"])
        storage = model_payload_storage(payload)
        # 每个矩阵的误差，避免只看整体 MSE 抹掉敏感层。
        weight_errors[name] = {key: error_metrics(model.state_dict()[key].cpu(), dequantize_weight(p))
                               for key, p in loaded["quantized_weights"].items()}
        result = {"variant": name, "loss": other["loss"], "accuracy": other["accuracy"],
                  **storage, **compare_predictions(base, other), "file_bytes": path.stat().st_size,
                  "fp32_runtime_parameter_bytes": sum(tensor_bytes(t) for t in reconstructed.state_dict().values()),
                  "serialized_reload_verified": True}
        results.append(result)
        raw["predictions"][name] = other["predictions"].tolist()
        raw["token_nll"][name] = other["token_nll"].tolist()
        del reconstructed
    if fingerprint(model.state_dict()) != source_hash:
        raise RuntimeError("源模型在评测中被改写")
    report = {"environment": {"python": platform.python_version(), "torch": str(torch.__version__),
                              "device": str(device), "cuda_available": torch.cuda.is_available(),
                              "threads": torch.get_num_threads(),
                              "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None},
              "protocol": {"seed": 3001, "steps": steps, "train_samples": len(data["train"]),
                           "test_samples": len(data["test"]), "test_tokens": base["targets"].numel(),
                           "source_sha256": source_hash, "model_config": asdict(model.config),
                           "parameter_count": sum(p.numel() for p in model.parameters()),
                           "target_weights": "所有 nn.Linear.weight，含 lm_head；不量化Embedding和Norm",
                           "selection": "固定训练步数和预设量化方案；无测试集调参",
                           "limits": "单种子合成token任务，不是真实模型评测；FP32反量化模拟，无低比特GEMM，无速度/峰值显存结论"},
              "results": results, "source_unchanged": True, "sources": SOURCES}
    write_json(out / "metrics.json", report)
    write_json(out / "weight_errors.json", weight_errors)
    write_json(out / "predictions.json", raw)
    write_json(out / "training.json", {"history": history, "train_ids": data["train"].tolist(), "test_ids": data["test"].tolist()})
    return report

In [20]:
RUN_MODEL_EXPERIMENT = True
DEVICE = "cpu"       # 可改成 "cuda"；CUDA不可用会报错。当前制作环境只实测CPU。
STEPS = 100
OUTPUT_DIR = Path("lesson_30_outputs")
report = None
if RUN_MODEL_EXPERIMENT:
    report = run_experiment(OUTPUT_DIR, STEPS, choose_device(DEVICE))
    print("运行完成。参数量:", report["protocol"]["parameter_count"])
    print("源模型未改写:", report["source_unchanged"])
else:
    print("已明确跳过完整训练；这不影响前面的数学实验。")

运行完成。参数量: 52368
源模型未改写: True


### 12.1 不要只盯着一个 loss 数字

下面同时看测试交叉熵、token准确率、预测变化和张量存储载荷。量化的细小扰动有时也会让这份小测试集的 loss 下降，这并不等于可靠增强了模型能力。

`changed_top1` 统计全部234个目标位置中，与原模型预测不同的位置。`fixed_tokens` 是原来错、现在对；`regressed_tokens` 是原来对、现在错。

第一步从 BOS 猜新符号有固有不确定性；不要要求这个任务所有位置都能达到100%。

In [21]:
if report is not None:
    print(f"{'方案':>16} {'loss':>9} {'准确率':>9} {'变化':>6} {'改对':>5} {'改错':>5} {'载荷B':>10} {'文件B':>10}")
    for row in report["results"]:
        print(f"{row['variant']:>16} {row['loss']:9.6f} {row['accuracy']:9.4f} "
              f"{row['changed_top1']:6d} {row['fixed_tokens']:5d} {row['regressed_tokens']:5d} "
              f"{row['total_tensor_payload_bytes']:10d} {row['file_bytes']:10d}")
    print("以上是单种子合成任务，不是真实大模型基准。")

              方案      loss       准确率     变化    改对    改错        载荷B        文件B
            fp32  0.774146    0.7607      0     0     0     209472     216912
     int8_tensor  0.773137    0.7607      0     0     0      55308      67359
    int8_channel  0.774950    0.7607      0     0     0      58892      70921
     int4_tensor  0.770003    0.7521     27     0     2      29604      41631
    int4_channel  0.769926    0.7607      2     1     1      33188      45193
    int4_group32  0.773079    0.7607      1     0     0      44120      56265
    int4_group16  0.778246    0.7564      2     0     1      42396      54409
以上是单种子合成任务，不是真实大模型基准。


### 12.2 较小 group 为什么不一定让这里的文件更大？

第6节比较的是没有补齐的 `[32,128]` 矩阵，此时小 group 会增加尺度开销。

完整模型中有许多输入宽度为 **48** 的矩阵：`group_size=32` 要补成64，`group_size=16` 不需要补齐。因此，两种格式的**码数量和尺度数量都会变化**，不能只比较尺度个数。

更不能认为 group 越小一定让每个权重或每道题更准确。格点变化、舍入、传播路径和任务边界都要实际核对。

下面逐项核对模型恢复后的 FP32 权重载荷，确认没有把“压缩文件”误当作“压缩执行”。

In [22]:
if report is not None:
    for row in report["results"]:
        print(row["variant"], "恢复后的FP32权重载荷:", row["fp32_runtime_parameter_bytes"], "B")
    sizes = {r["fp32_runtime_parameter_bytes"] for r in report["results"]}
    assert len(sizes) == 1
    print("所有参考执行模型的FP32权重载荷相同。未测峰值内存，不给加速倍数。")

fp32 恢复后的FP32权重载荷: 209472 B
int8_tensor 恢复后的FP32权重载荷: 209472 B
int8_channel 恢复后的FP32权重载荷: 209472 B
int4_tensor 恢复后的FP32权重载荷: 209472 B
int4_channel 恢复后的FP32权重载荷: 209472 B
int4_group32 恢复后的FP32权重载荷: 209472 B
int4_group16 恢复后的FP32权重载荷: 209472 B
所有参考执行模型的FP32权重载荷相同。未测峰值内存，不给加速倍数。


## 13. 脱离训练过程，重新计算指标

沿用第29课的原则：原始记录必须足够复核汇总，不只保存一个漂亮的总分。

下面从 `predictions.json` 重新计算每种方案的准确率和平均负对数似然，核对报告。总分相同也可能存在不同位置的修复和退步。

In [23]:
if report is not None:
    raw = json.loads((OUTPUT_DIR / "predictions.json").read_text(encoding="utf-8"))
    target = torch.tensor(raw["targets"])
    for row in report["results"]:
        name = row["variant"]
        prediction = torch.tensor(raw["predictions"][name])
        recovered_acc = float((prediction == target).float().mean())
        recovered_loss = float(torch.tensor(raw["token_nll"][name]).mean())
        assert abs(recovered_acc - row["accuracy"]) < 1e-6
        assert abs(recovered_loss - row["loss"]) < 1e-6
    print("全部方案：原始记录重评分与报告一致。")

全部方案：原始记录重评分与报告一致。


### 13.1 重新加载压缩文件

下面不再训练，读取本次生成的 `int4_group16.pt`，恢复模型并再次评测。文件保存的不仅是四位码，还包含每层形状、分组和尺度。

`weights_only=True` 限制允许反序列化的对象，但仍然只应加载可信文件；它不是“只恢复一个叫 weight 的字段”。

In [24]:
if report is not None:
    loaded_payload = torch.load(OUTPUT_DIR / "int4_group16.pt", map_location="cpu", weights_only=True)
    reloaded_model = restore_model(loaded_payload, choose_device(DEVICE))
    reloaded_result = evaluate(reloaded_model, make_data()["test"])
    expected = next(r for r in report["results"] if r["variant"] == "int4_group16")
    assert abs(reloaded_result["loss"] - expected["loss"]) < 1e-6
    print("重载loss:", reloaded_result["loss"])
    print("恢复后的参数dtype:", {p.dtype for p in reloaded_model.parameters()})
    del reloaded_model

重载loss: 0.7782463431358337
恢复后的参数dtype: {torch.float32}


## 14. 独立练习 A：每输出行的对称量化

实现 `quantize_rows_ex(w,bits)`，返回 `(q,scales,reconstructed)`：

- `w` 是 `[out,in]` 的 FP32 权重；每行单独取 absmax。
- 本课 INT4 用 `[-7,7]`，INT8 用 `[-127,127]`。
- `q` 为 int8，形状与 w 相同；`scales` 为 `[out,1]` 的 FP32。
- 全零行用尺度1；不修改 w。
- 这是 PTQ，不要求量化操作可微。

不要直接调用现成的 `quantize_weight()` 返回答案。完成后把开关改成 `True`。

In [25]:
def exercise_a_checks(fn):
    w = torch.tensor([[0.0, 0.2, -0.4], [3.0, -6.0, 0.0], [0.0, 0.0, 0.0]])
    old = w.clone()
    for bits in (4, 8):
        q, scales, restored = fn(w, bits)
        p = quantize_weight(w, bits, "channel", None)
        torch.testing.assert_close(q, p["codes"].reshape_as(w))
        torch.testing.assert_close(scales, p["scales"].reshape(-1, 1))
        torch.testing.assert_close(restored, dequantize_weight(p))
    torch.testing.assert_close(w, old, rtol=0, atol=0)
    print("练习A通过：每输出行尺度、全零行与源权重保护")


def exercise_b_checks(pack_fn, unpack_fn):
    for values in ([], [-8], [7], [-8, -1, 0, 7, 2], list(range(-8, 8))):
        q = torch.tensor(values, dtype=torch.int8)
        packed = pack_fn(q)
        torch.testing.assert_close(packed, pack_int4(q), rtol=0, atol=0)
        torch.testing.assert_close(unpack_fn(packed, len(values)), q, rtol=0, atol=0)
    print("练习B通过：全部16种码、奇数末尾和低/高4位顺序")


def exercise_c_checks(fn):
    x = torch.tensor([[1.0, 2.0, -3.0], [-1.0, 0.5, 4.0]])
    w = torch.tensor([[0.1, -0.2, 0.7], [5.0, 0.3, -2.0]])
    bias = torch.tensor([0.25, -1.0])
    p = pack_packet(quantize_weight(w, 4, "group", 2))
    expected = F.linear(x, dequantize_weight(p), bias)
    torch.testing.assert_close(fn(x, p, bias), expected)
    torch.testing.assert_close(fn(x, p, None), F.linear(x, dequantize_weight(p)))
    x3 = x.reshape(1, 2, 3)
    torch.testing.assert_close(fn(x3, p, bias), expected.reshape(1, 2, 2))
    print("练习C通过：还原分组、裁掉补齐列、Linear权重方向与bias")

In [26]:
def quantize_rows_ex(w, bits):
    raise NotImplementedError("自己写：逐行absmax、尺度、舍入、截断、重建")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    exercise_a_checks(quantize_rows_ex)
else:
    print("练习A尚未执行；完成后开启检查。")

练习A尚未执行；完成后开启检查。


## 15. 独立练习 B：真正打包与解包 INT4

输入已经保证是合法 `[-8,7]` 的整数码。实现两函数：

- `pack_ex(q)`：先做 `+8`，先出现的码放低4位，奇数高4位补8。
- `unpack_ex(packed,count)`：提取两组 nibble，交错还原顺序，裁掉最后填充，再减8。

核心表达式可以使用 `& 15`、`>> 4`、`<< 4`、`|`。先用纸笔解释 `[-7,-1] → 113`，再写循环或张量版本。

不要调用已提供的 `pack_int4/unpack_int4` 来绕过实现。

In [27]:
def pack_ex(q):
    raise NotImplementedError("自己写偏移编码和低/高位合并")

def unpack_ex(packed, count):
    raise NotImplementedError("自己写解包、恢复顺序、减8")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    exercise_b_checks(pack_ex, unpack_ex)
else:
    print("练习B尚未执行；完成后开启检查。")

练习B尚未执行；完成后开启检查。


## 16. 独立练习 C：把存储格式接回 Linear

实现 `quantized_linear_reference_ex(x,packet,bias=None)`：

1. 使用 `dequantize_weight()` 恢复正确形状。
2. 让恢复的权重与 x 使用同一 device/dtype。
3. 计算 `x @ W.T`，可选加 bias，支持 `[B,T,in_features]` 输入。

**要能口头解释：为什么这个函数没有低比特计算加速？** 因为它恢复了浮点权重，然后调用浮点矩阵运算。

本题允许调用已有的反量化工具，不要求重写文件格式验证。

In [28]:
def quantized_linear_reference_ex(x, packet, bias=None):
    raise NotImplementedError("恢复权重，并注意Linear的转置与bias")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    exercise_c_checks(quantized_linear_reference_ex)
else:
    print("练习C尚未执行；完成后开启检查。")

练习C尚未执行；完成后开启检查。


---
## 17. 参考答案：请先独立完成练习

以下函数采用不同名称，不会覆盖你的实现。它们只针对练习约定的合法输入；完整接口的错误处理请看前面的生产式校验。

In [29]:
def answer_a(w, bits):
    finite_float(w)
    limit = bit_limit(bits)
    max_abs = w.detach().float().abs().amax(-1, keepdim=True)
    scales = torch.where(max_abs == 0, torch.ones_like(max_abs), max_abs / limit)
    q = (w.detach().float() / scales).round().clamp(-limit, limit).to(torch.int8)
    return q, scales, q.float() * scales

exercise_a_checks(answer_a)

练习A通过：每输出行尺度、全零行与源权重保护


In [30]:
def answer_b_pack(q):
    u = (q.to(torch.int16).flatten() + 8).to(torch.uint8)
    if len(u) % 2:
        u = torch.cat([u, torch.tensor([8], device=u.device, dtype=torch.uint8)])
    return u[::2] | (u[1::2] << 4)


def answer_b_unpack(p, n):
    two = torch.stack([p & 15, p >> 4], dim=-1).flatten()[:n]
    return (two.to(torch.int16) - 8).to(torch.int8)

exercise_b_checks(answer_b_pack, answer_b_unpack)

练习B通过：全部16种码、奇数末尾和低/高4位顺序


In [31]:
def answer_c(x, packet, bias=None):
    restored = dequantize_weight(packet).to(device=x.device, dtype=x.dtype)
    return F.linear(x, restored, bias)

exercise_c_checks(answer_c)

练习C通过：还原分组、裁掉补齐列、Linear权重方向与bias


## 18. 自动检查：证明实现遵守哪些约定

下面检查手算值、零点、官方 INT8 接口、通道方向、非连续矩阵、打包奇偶长度、分组填充、文件重载、源权重保护、配对指标和模型因果性。

检查数量由实际成功的检查名称累计，不把一次训练循环中的每个 token 算成一项测试。

你还可以主动制造故障：交换打包高低位、去掉补齐裁剪、把零点相加改成相减，再运行检查。**测试应报告失败，而不是继续打印全部通过。**

In [32]:
class Checks:
    def __init__(self): self.names = []
    def ok(self, condition, name):
        if not bool(condition): raise AssertionError(name)
        self.names.append(name)
    def close(self, a, b, name, **kwargs):
        torch.testing.assert_close(a, b, **kwargs)
        self.names.append(name)
    def rejects(self, fn, name):
        try: fn()
        except (ValueError, TypeError, KeyError): self.names.append(name); return
        raise AssertionError("未拒绝：" + name)


def run_checks():
    c = Checks()
    # 基础公式与数值边界。
    x = torch.tensor([-1.0, -0.1, 0.0, 0.1, 1.0])
    q, s, r = symmetric_vector(x, 4)
    c.close(q, torch.tensor([-7, -1, 0, 1, 7], dtype=torch.int32), "手算INT4码")
    c.ok(s == 1 / 7, "手算尺度")
    c.ok(float((r - x).abs().max()) <= s / 2 + 1e-7, "无裁剪误差界")
    c.close(torch.round(torch.tensor([-2.5, -1.5, -.5, .5, 1.5, 2.5])), torch.tensor([-2., -2., 0., 0., 2., 2.]), "ties-even")
    c.close(affine_quantize(torch.tensor([-10., 0., 10.]), .1, 0, -7, 7), torch.tensor([-7, 0, 7], dtype=torch.int32), "先clamp再整数存储")
    c.close(affine_quantize(torch.tensor([-.2, 0., .3]), .1, 5, 0, 15), torch.tensor([3, 5, 8], dtype=torch.int32), "仿射zero-point")
    c.close(affine_dequantize(torch.tensor([5]), .1, 5), torch.tensor([0.]), "零点重建零")
    c.ok(affine_params(torch.zeros(3)) == (1., 0), "全零仿射参数")
    c.ok(affine_params(torch.tensor([2., 3.]))[1] == 0, "全正范围包含零")
    c.ok(affine_params(torch.tensor([-3., -2.]))[1] == 15, "全负范围包含零")
    c.rejects(lambda: affine_quantize(x, 0), "拒绝零尺度")
    c.rejects(lambda: affine_quantize(x, float('nan')), "拒绝NaN尺度")
    c.rejects(lambda: symmetric_vector(torch.tensor([float('inf')])), "拒绝无穷输入")
    c.rejects(lambda: bit_limit(True), "拒绝布尔bits")
    c.rejects(lambda: symmetric_vector(x, 3), "拒绝未实现位数")
    c.rejects(lambda: symmetric_vector(torch.empty(0)), "拒绝空输入")
    # 与官方CPU qint8核对；测试点刻意避开浮点的半整数边界。
    ref = torch.tensor([-0.93, -0.22, 0., 0.31, .89])
    official = torch.quantize_per_tensor(ref, 0.1, 0, torch.qint8)
    c.close(affine_quantize(ref, .1, 0, -128, 127).to(torch.int8), official.int_repr(), "官方per-tensor整码")
    c.close(affine_dequantize(official.int_repr(), .1), official.dequantize(), "官方per-tensor重建")
    w = torch.tensor([[0., 0., 0., 0.], [.11, -.23, .37, -.53], [10., -20., 30., -40.]])
    old = w.clone()
    for bits in (4, 8):
        for scope, group in (("tensor", None), ("channel", None), ("group", 3)):
            p = quantize_weight(w, bits, scope, group)
            packed = pack_packet(p)
            a, b = dequantize_weight(p), dequantize_weight(packed)
            c.close(a, b, f"{bits}-{scope}打包重建一致", rtol=0, atol=0)
            c.ok(a.shape == w.shape, f"{bits}-{scope}恢复形状")
            c.ok(bool(torch.isfinite(a).all()), f"{bits}-{scope}有限")
            n = p["codes"].numel()
            c.ok(tensor_bytes(packed["codes"]) == ((n + 1) // 2 if bits == 4 else n), f"{bits}-{scope}真实码字节")
    c.close(w, old, "源矩阵不变", rtol=0, atol=0)
    p = quantize_weight(w, 8, "channel", None)
    official = torch.quantize_per_channel(w, p["scales"].double(), torch.zeros(3, dtype=torch.int64), 0, torch.qint8)
    c.close(p["codes"].reshape_as(w), official.int_repr(), "官方per-channel整码")
    c.close(dequantize_weight(p), official.dequantize(), "官方per-channel输出")
    c.ok(float(p["scales"][0]) == 1., "全零行安全尺度")
    c.close(dequantize_weight(quantize_weight(w.T, 4, "group", 2)), dequantize_weight(quantize_weight(w.T.contiguous(), 4, "group", 2)), "非连续权重")
    c.rejects(lambda: quantize_weight(torch.ones(3)), "拒绝非矩阵")
    c.rejects(lambda: quantize_weight(torch.ones(2, 2), 4, "group", 0), "拒绝非法group")
    c.rejects(lambda: quantize_weight(torch.tensor([[float('nan')]])), "拒绝NaN权重")
    # 打包所有16种码，含对称量化不使用的-8；打包函数本身支持完整物理范围。
    for n in (0, 1, 2, 5, 16, 31):
        codes = ((torch.arange(n) % 16) - 8).to(torch.int8)
        c.close(unpack_int4(pack_int4(codes), n), codes, f"打包往返{n}", rtol=0, atol=0)
    c.close(pack_int4(torch.tensor([-7, -1, 0, 7, 3], dtype=torch.int8)), torch.tensor([113, 248, 139], dtype=torch.uint8), "打包低高位手算")
    c.rejects(lambda: pack_int4(torch.tensor([8], dtype=torch.int8)), "拒绝越界码")
    c.rejects(lambda: unpack_int4(torch.tensor([0], dtype=torch.uint8), 1), "拒绝奇数尾填充错误")
    c.rejects(lambda: unpack_int4(torch.tensor([128], dtype=torch.uint8), 3), "拒绝长度错误")
    bad = copy.deepcopy(p); bad["scales"][0] = 0
    c.rejects(lambda: dequantize_weight(bad), "拒绝损坏尺度")
    bad = copy.deepcopy(p); bad["shape"] = [4, 4]
    c.rejects(lambda: dequantize_weight(bad), "拒绝损坏shape")
    z = pack_packet(quantize_weight(torch.ones(1, 3), 4, "group", 2))
    z["codes"][-1] = 255
    c.rejects(lambda: dequantize_weight(z), "拒绝组内非零填充")
    nums = numeric_experiments()
    c.ok(nums["row_scale_demo"]["channel"]["error"]["mse"] < nums["row_scale_demo"]["tensor"]["error"]["mse"], "通道量级反例")
    c.ok(nums["storage"]["group_32"]["packed"]["tensor_payload_bytes"] == 2560, "含scale载荷2560字节")
    c.ok(nums["storage"]["group_16"]["packed"]["tensor_payload_bytes"] == 3072, "小组元数据增加")
    c.ok(nums["storage"]["group_32"]["unpacked"]["code_bytes"] == 4096, "未打包INT4实际1字节")
    cd = calibration_demo(); c.ok(cd["selected_by_calibration_only"] == 1, "只按校准集选阈值")
    c.ok(cd["candidates"][1]["shift_test_mse"] > cd["candidates"][0]["shift_test_mse"], "分布偏移反例")
    c.ok(cd["candidates"][1]["weight_mse"] > cd["candidates"][0]["weight_mse"], "权重MSE与输入输出误差不同")
    # 全模型量化、文件往返、无副作用和最小反向。
    model = new_model(); model.eval()
    rows = make_data()["test"][:3]
    before = fingerprint(model.state_dict())
    c.ok(restore_model(pack_model(model, 8, "channel")).config == model.config, "per-channel默认调用兼容")
    for bits, scope, group in ((8, "channel", None), (4, "group", 16)):
        payload = pack_model(model, bits, scope, group)
        restored = restore_model(payload)
        result = evaluate(restored, rows)
        c.ok(math.isfinite(result["loss"]), f"全模型{bits}loss有限")
        c.ok(all(t.dtype == torch.float32 for t in restored.parameters()), f"全模型{bits}真实计算仍FP32")
        c.close(restored.embedding.weight, model.embedding.weight, f"{bits}Embedding不量化", rtol=0, atol=0)
        c.close(restored.norm.weight, model.norm.weight, f"{bits}Norm不量化", rtol=0, atol=0)
        c.ok(sum(tensor_bytes(t) for t in restored.state_dict().values()) == sum(tensor_bytes(t) for t in model.state_dict().values()), f"{bits}FP32运行权重未减少")
        with tempfile.TemporaryDirectory() as td:
            path = Path(td) / "quant.pt"; torch.save(payload, path)
            reloaded = restore_model(torch.load(path, map_location="cpu", weights_only=True))
            c.close(evaluate(reloaded, rows)["logits"], result["logits"], f"{bits}文件重载输出", rtol=0, atol=0)
        c.ok(model_payload_storage(payload)["total_tensor_payload_bytes"] < sum(tensor_bytes(t) for t in model.state_dict().values()), f"{bits}存储载荷减少")
    c.ok(before == fingerprint(model.state_dict()), "全模型源权重不变")
    bad = copy.deepcopy(payload); del bad["quantized_weights"]["head.weight"]
    c.rejects(lambda: restore_model(bad), "拒绝缺少量化层")
    bad = copy.deepcopy(payload); bad["quantization"]["bits"] = 8
    c.rejects(lambda: restore_model(bad), "拒绝不一致元数据")
    bad = copy.deepcopy(payload); bad["format"] = "other"
    c.rejects(lambda: restore_model(bad), "拒绝外部不兼容格式")
    a = evaluate(model, rows)
    c.ok(compare_predictions(a, a)["changed_top1"] == 0, "自比较不变")
    mutated = copy.deepcopy(a); mutated["targets"][0, 0] += 1
    c.rejects(lambda: compare_predictions(a, mutated), "拒绝不同目标配对")
    with torch.no_grad():
        ids = rows[:, :-1]; altered = ids.clone(); altered[:, -1] = 3
        c.close(model(ids)[:, :-1], model(altered)[:, :-1], "模型因果性", rtol=0, atol=0)
    # PTQ不是训练算子；直接round没有STE。
    v = torch.tensor([.2, .7], requires_grad=True); torch.round(v).sum().backward()
    c.close(v.grad, torch.zeros_like(v), "round梯度不是STE")
    c.ok(len(make_data()["train"]) == 106 and len(make_data()["test"]) == 26, "固定切分")
    c.ok(not {tuple(x) for x in make_data()["train"].tolist()} & {tuple(x) for x in make_data()["test"].tolist()}, "训练测试样本不重合")
    return {"passed": len(c.names), "names": c.names}

In [33]:
check_result = run_checks()
print("固定检查通过:", check_result["passed"], "项")
print("参考答案检查与固定检查分开记录；上面的练习仍须自己完成。")

固定检查通过: 95 项
参考答案检查与固定检查分开记录；上面的练习仍须自己完成。


## 19. 独立脚本与文件说明

```bash
# 只跑数学和边界检查，不训练
python lesson_30_weight_quantization_int8_int4.py --checks-only

# 数学、离群值、打包与载荷实验
python lesson_30_weight_quantization_int8_int4.py --math-only

# 默认训练一次，再比较全部预定量化方案
python lesson_30_weight_quantization_int8_int4.py

# 已有本课压缩文件：仅重载、评测，不训练，不写文件
python lesson_30_weight_quantization_int8_int4.py --load-packed lesson_30_outputs/int4_group16.pt
```

默认输出 `lesson_30_outputs`，包含 `source_fp32.pt`、六个量化文件、`metrics.json`、`numerics.json`、`weight_errors.json`、`predictions.json`、`training.json`。同名生成文件会覆盖。

`--device cuda` 可让训练和 FP32 参考评测使用你的显卡，前提是现有环境可调用 CUDA；这**不会**把参考实现变成 INT4 GPU 内核。量化统计与打包始终在 CPU 上完成。

这些文件不保存优化器、数据游标或 RNG，不支持精确断点续训；也不能作为真实模型的通用量化转换器。

本次默认完整运行在 CPU 验证。没有 CUDA 峰值显存、低比特吞吐或 Qwen 的量化成绩。

## 20. 本课验收

完成本课时，你应该能拿着实际数据解释：

- 给定 `scale、zero_point、qmin/qmax`，怎样从小数得到码、再重建近似值。
- 为什么本课 INT4 只用15个对称值，而打包器能表示16种物理码。
- 为什么一个码值很小的 `int8` 张量，仍然占一个字节。
- 为什么同一个权重矩阵的误差，需要结合输入、logits和任务指标一起判断。
- 为什么本课量化文件变小，但参考推理的权重张量仍与FP32基线一样大。
- 为什么某组 loss 更低，并不能据此宣称量化提升了通用模型能力。

**下一课：第 31 课——真实小模型的低比特加载与回归评测。** 在环境探测成功后，使用实际量化后端，核对真实模块、权重格式和运行资源，再接回第29课的固定题集。

## 来源与验证边界

以下均为官方文档、原始研究或项目自身资料；网页内容可能随版本更新。运行报告记录本机 PyTorch 版本。

1. [Hugging Face：Quantization concepts](https://huggingface.co/docs/transformers/quantization/concept_guide) — 量化格式、打包与PTQ/QAT。
2. [LLVM/MLIR：Quantization](https://mlir.llvm.org/docs/Quantization/) — scale、zero point、仿射映射及舍入误差。
3. [PyTorch：torch.round](https://docs.pytorch.org/docs/2.10/generated/torch.round.html) — 半整数舍入约定。
4. [PyTorch：quantize_per_tensor](https://docs.pytorch.org/docs/2.10/generated/torch.quantize_per_tensor.html)、[quantize_per_channel](https://docs.pytorch.org/docs/2.10/generated/torch.quantize_per_channel.html) — 官方INT8参考接口。
5. [PyTorch：QAT说明](https://pytorch.org/blog/quantization-aware-training/) — 量化模拟与STE，区别于直接整数存储。
6. [Hugging Face：bitsandbytes](https://huggingface.co/docs/transformers/quantization/bitsandbytes) — NF4/FP4和计算精度；本课没有调用该后端。
7. [GPTQ](https://arxiv.org/abs/2210.17323)、[AWQ](https://arxiv.org/abs/2306.00978) — 为什么实际低比特方法会进一步利用输入与误差信息；本课未复现这两种算法。

数学例子、载荷表和合成模型结果来自本讲义代码的实际执行，不是上述论文中的性能数据。